# 05 — Results Analysis & Comparison

**Goal:** Compare Isolation Forest vs TGN on the 3W dataset and produce thesis-ready visualizations.

## Outputs:
1. Metrics comparison table (F1, Precision, Recall, AUC-ROC)
2. Evidence chain example (root cause analysis via Cypher)
3. Temporal anomaly visualization per event type
4. Discussion of TKG advantages vs flat models

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from neo4j import GraphDatabase
import warnings
warnings.filterwarnings('ignore')
import sys
sys.path.append('../../src')
from config import DATA_ROOT_3W, PROCESSED_DIR

NEO4J_URI      = 'bolt://172.22.43.151:7687'
NEO4J_USER     = 'neo4j'
NEO4J_PASSWORD = 'your_password'
EXPERIMENTS_DIR = Path('../../experiments')
EXPERIMENTS_DIR.mkdir(exist_ok=True)

EVENT_TYPES = {
    0:'Normal', 1:'Abrupt BSW', 2:'Spurious DHSV', 3:'Severe Slugging',
    4:'Flow Instability', 5:'Rapid Productivity Loss',
    6:'Quick Restriction PCK', 7:'Scaling PCK', 8:'Hydrate'
}

print('✅ Setup complete')

## 1. Metrics Comparison Table

In [ ]:
# Fill these after running the models
# Use Case 1 results (synthetic data — already computed)
uc1_results = {
    'Isolation Forest (UC1 synthetic)': {'F1_anomaly': 0.921, 'AUC_ROC': 0.991, 'Precision': 0.974, 'Recall': 0.874},
    'TGN (UC1 synthetic)':              {'F1_anomaly': 0.615, 'AUC_ROC': 1.000, 'Precision': 0.444, 'Recall': 1.000},
}

# Use Case 2 results (3W real data — fill after training)
uc2_results = {
    'Isolation Forest (UC2 3W)': {'F1_anomaly': None, 'AUC_ROC': None, 'Precision': None, 'Recall': None},
    'TGN (UC2 3W)':              {'F1_anomaly': None, 'AUC_ROC': None, 'Precision': None, 'Recall': None},
}

all_results = {**uc1_results, **uc2_results}
results_df = pd.DataFrame(all_results).T
print('Results Summary:')
print(results_df.to_string())

## 2. Evidence Chain — Root Cause Analysis

In [ ]:
def get_evidence_chain(well_id: str, sensor_id: str, timestamp: str) -> list:
    """
    Traces the causal chain for an anomaly via Cypher.
    This is the key TKG advantage over flat models — explainability.
    
    Ref: Khan et al. [9] — rule-based models provide transparent, auditable reasoning
    """
    driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
    with driver.session() as session:
        result = session.run('''
            MATCH (w:Well {id: $well_id})
                  -[:HAS_SENSOR]->(s:Sensor {id: $sensor_id})
                  -[:MADE_OBSERVATION]->(o:Observation)
                  -[:DETECTED_ANOMALY]->(a:AnomalyEvent)
            WHERE o.valid_from >= $ts_start
            AND   o.valid_from <= $ts_end
            RETURN w.id AS well,
                   s.id AS sensor,
                   s.position AS position,
                   o.value AS value,
                   o.valid_from AS timestamp,
                   a.anomaly_type AS anomaly_type
            ORDER BY o.valid_from
            LIMIT 10
        ''', well_id=well_id, sensor_id=sensor_id,
             ts_start=timestamp[:10] + 'T00:00:00',
             ts_end=timestamp[:10] + 'T23:59:59')
        chain = [dict(r) for r in result]
    driver.close()
    return chain

print('Evidence chain function ready.')
print('Usage: get_evidence_chain("WELL-00001", "P-PDG", "2016-01-15")')

## 3. Visualization — Metrics Comparison

In [ ]:
def plot_metrics_comparison(results: dict):
    """Bar chart comparison of all models across metrics."""
    metrics = ['F1_anomaly', 'AUC_ROC', 'Precision', 'Recall']
    models  = list(results.keys())
    colors  = ['#89b4fa', '#a6e3a1', '#f9e2af', '#f38ba8']

    fig, axes = plt.subplots(1, len(metrics), figsize=(16, 5))
    fig.patch.set_facecolor('#181825')

    for ax, metric, color in zip(axes, metrics, colors):
        values = [results[m].get(metric) for m in models]
        valid  = [(m, v) for m, v in zip(models, values) if v is not None]

        if valid:
            names, vals = zip(*valid)
            bars = ax.bar(range(len(names)), vals, color=color, edgecolor='#313244', alpha=0.85)
            for bar, val in zip(bars, vals):
                ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                       f'{val:.3f}', ha='center', va='bottom', fontsize=8, color='#cdd6f4')
            ax.set_xticks(range(len(names)))
            ax.set_xticklabels([n.split('(')[0].strip() for n in names],
                               rotation=45, ha='right', fontsize=8)

        ax.set_title(metric, color='#cdd6f4', fontsize=10)
        ax.set_ylim(0, 1.1)
        ax.set_facecolor('#1e1e2e')
        ax.tick_params(colors='#cdd6f4')
        ax.spines['bottom'].set_color('#313244')
        ax.spines['left'].set_color('#313244')

    fig.suptitle('Model Comparison — Use Case 1 (Synthetic) vs Use Case 2 (3W Real)',
                color='#cba6f7', fontsize=12)
    plt.tight_layout()
    plt.savefig(EXPERIMENTS_DIR / 'metrics_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'✅ Saved to {EXPERIMENTS_DIR}/metrics_comparison.png')

plot_metrics_comparison(all_results)

## 4. TKG Advantages Summary

| Feature | Flat Model (IF) | TKG + TGN |
|---|---|---|
| Anomaly detection | ✅ | ✅ |
| Root cause explanation | ❌ | ✅ via Cypher chain |
| Multi-sensor context | ❌ | ✅ via PRECEDES |
| Cross-well generalization | ❌ | ✅ via CAUSALLY_COUPLED |
| Auditability (IEC 61508) | ❌ | ✅ bitemporal trace |
| Missing value handling | imputation | graph neighbor inference |
| Temporal windows | fixed | dynamic per event type |